# 🏭 Celeris Automobili Industrial: Demand Forecasting & Planning Control Tower
### MKS AIML Hackathon Phase 2: Production End-to-End Enterprise Demand Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pkprajapati7402/mks-hackathon/blob/main/mks_datathon_notebook.ipynb)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-3776AB?logo=python&logoColor=white)](https://www.python.org/)
[![LightGBM](https://img.shields.io/badge/LightGBM-4.3%2B-brightgreen)](https://lightgbm.readthedocs.io/)
[![19-Month Forecast](https://img.shields.io/badge/Forecast%20Horizon-Nov%202026%20--%20May%202028-orange)](#)
[![Submission Integrity](https://img.shields.io/badge/Submission%20Rows-58%2C219%20Valid-success)](#)

---
### 📌 Executive Overview
This interactive notebook demonstrates the enterprise demand forecasting pipeline and S&OP control tower built for **Celeris Automobili Industrial** covering **601 commercial SKUs, 7 product families, 4 channels, and 6 global regions**.

#### 🎯 Key Technical Capabilities Demonstrated:
1. **Systematic ERP Data Cleaning**: 15+ vectorized business rules correcting SKU typos, customer name aliases, regional consolidations, negative returns, and year transposition errors.
2. **True Unconstrained Demand Reconstruction**: Correcting historical sales for inventory stockouts with customer-specific proportional allocation to prevent carrying supply-chain bottlenecks into future build schedules.
3. **Anomaly Intelligence Register**: Categorizing and explaining 2,100 historical structural anomalies (supply shocks, early buying spikes before price hikes, pandemic dips).
4. **95-Feature Panel & Macro Econometric Drivers**: Cross-correlation lead/lag analysis with Automotive Production and Steel indices, plus quantified EV vs. ICE transition dynamics.
5. **Leakage-Free 19-Month Rolling Backtesting**: Evaluating Seasonal Naive, Moving Average, Intermittent Croston SBA, and Multi-Quantile LightGBM across multiple origins (`2023-10`, `2024-10`, `2025-03` holdout).
6. **Exact Hierarchical Reconciliation & Life-Cycle Governance**: Bottom-up aggregation ensuring $\Delta = 0$ discrepancy across SKU, Family, Region, and Business Unit hierarchies, with zero demand post-EOL.

## 1. Environment Verification & Auto-Setup (Google Colab & Local)
Automatically detects the runtime environment, clones/pulls the repository if running inside Google Colab, handles directory switching, installs dependencies, and configures native Plotly rendering.

In [ ]:
# -----------------------------------------------------------------------------
# 1. ROBUST ENVIRONMENT DETECTION & AUTO-SETUP (COLAB + LOCAL)
# -----------------------------------------------------------------------------
import os
import sys
import shutil
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/pkprajapati7402/mks-hackathon.git"
REPO_NAME = "mks-hackathon"

if IN_COLAB:
    print("⚡ Running in Google Colab environment.")
    
    # 1. Check if src/ exists in current working directory
    if not (Path.cwd() / "src").exists():
        colab_repo_path = Path(f"/content/{REPO_NAME}")
        
        # If folder exists but has no src/ (stale or partial clone), wipe and re-clone
        if colab_repo_path.exists() and not (colab_repo_path / "src").exists():
            print("⚠️ Removing outdated repository clone...")
            shutil.rmtree(str(colab_repo_path), ignore_errors=True)
        
        # Clone repository if needed
        if not colab_repo_path.exists():
            print(f"📥 Cloning full production repository from {REPO_URL}...")
            subprocess.run(["git", "clone", REPO_URL, str(colab_repo_path)], check=True)
        
        # Navigate into repository root
        os.chdir(str(colab_repo_path))
        print(f"📁 Working directory set to: {Path.cwd()}")
        
        # Pull latest commits
        try:
            subprocess.run(["git", "pull"], capture_output=True)
        except Exception:
            pass
    
    # 2. Install production dependencies silently
    print("📦 Installing required libraries (pyarrow, lightgbm, streamlit, plotly, statsmodels)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "pyarrow>=15.0.0", "lightgbm>=4.3.0", "streamlit>=1.35.0",
        "plotly>=5.20.0", "statsmodels>=0.14.0", "scikit-learn>=1.4.0"
    ], check=True)
else:
    print(f"💻 Running in Local environment: {Path.cwd()}")

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Configure Plotly to render properly in Colab and Jupyter
import plotly.io as pio
pio.renderers.default = "colab" if IN_COLAB else "notebook_connected"

# Safe display function across notebook and CLI environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Core scientific imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import json

print(f"\n✅ Environment Ready!")
print(f"   - Python:          {sys.version.split()[0]}")
print(f"   - Project Root:    {PROJECT_ROOT}")
print(f"   - Plotly Renderer: {pio.renderers.default}")

## 2. Execute Master Demand Pipeline
Run the master one-command production pipeline to execute the complete 6-stage lifecycle:
- **Stage 1**: Clean raw ERP datasets & log row-level audit trails.
- **Stage 2**: Reconstruct unconstrained customer demand & detect anomalies.
- **Stage 3**: Assemble 95-feature panel & econometric macro lag features.
- **Stage 4**: Execute rolling-origin 19-month backtest tournament across origins.
- **Stage 5**: Train Multi-Quantile LightGBM ($P10, P50, P90$) and reconcile hierarchies.
- **Stage 6**: Generate AI/ML Model Card and Executive S&OP Summary.

In [ ]:
from run_pipeline import run_pipeline

# Execute end-to-end demand forecasting engine
# (Runs in ~70s from scratch; generates clean datasets, feature store, models, and submission)
run_pipeline(stage="all")

## 3. Data Integrity & Anomaly Register Analysis
Explore the row-level audit trail, data quality report, and categorized historical anomalies.

In [ ]:
from src.config import REPORTS_DIR, CLEANED_DATA_DIR

# 1. Load Data Quality Report
with open(REPORTS_DIR / "data_quality_report.json", "r", encoding="utf-8") as f:
    dq_report = json.load(f)

# Display file cleaning summary table
file_summaries = []
for fname, stats in dq_report.get("file_summaries", {}).items():
    file_summaries.append({
        "Dataset": fname,
        "Raw Rows": f"{stats.get('raw_rows', 0):,}",
        "Cleaned Rows": f"{stats.get('clean_rows', 0):,}",
        "Status": "Verified Clean ✅"
    })
dq_df = pd.DataFrame(file_summaries)
display(dq_df)

# 2. Inspect Anomaly Register
anomaly_df = pd.read_csv(REPORTS_DIR / "anomaly_register.csv")
print(f"\nTotal Anomalies Categorized: {len(anomaly_df):,}")
print("\nAnomaly Breakdown by Category:")
display(anomaly_df["Category"].value_counts().to_frame("Count"))

print("\nSample Structural Anomalies with Business Context:")
display(anomaly_df[["Date", "SKU_ID", "Region", "Category", "Actual_Units", "Severity", "Explanation"]].head(8))

## 4. Econometric Drivers & EV Powertrain Paradigm Shift
Analyse the cross-correlation of macroeconomic indicators (Automotive Production Index, Steel Price Index) with product demand, and quantify the transition from internal combustion engines (ICE) to electric vehicles (EV).

In [ ]:
# 1. Macro Lead/Lag Cross-Correlation Analysis
macro_lag_df = pd.read_csv(REPORTS_DIR / "macro_lead_lag_analysis.csv")
print("Macro Lead/Lag Cross-Correlation (Lags 0 to 6 months):")
display(macro_lag_df.head(10))

# 2. EV vs ICE Powertrain Shift Analysis
ev_df = pd.read_csv(REPORTS_DIR / "ev_shift_analysis.csv")
display(ev_df)

# Visualize the EV Transition Trajectory
fig_ev = px.bar(
    ev_df,
    x="Year",
    y=["ICE_Units", "EV_Units"],
    barmode="group",
    title="Powertrain Paradigm Shift: Annual Volume (ICE vs EV Components)",
    labels={"value": "Annual Shipped Units", "variable": "Powertrain Type"},
    color_discrete_map={"ICE_Units": "#EF4444", "EV_Units": "#10B981"},
    template="plotly_dark"
)
fig_ev.update_layout(paper_bgcolor="#1E293B", plot_bgcolor="#0F172A")
fig_ev.show()

## 5. Rolling-Origin Backtest Tournament & Champion Selection
Review 19-month out-of-sample backtest metrics across time-series cross-validation origins (`2023-10`, `2024-10`, `2025-03` holdout) comparing LightGBM against baseline and intermittent models.

In [ ]:
from src.config import OUTPUTS_DIR

metrics_df = pd.read_csv(OUTPUTS_DIR / "backtest_metrics.csv")
print("Rolling-Origin 19-Month Backtest Performance:")
display(metrics_df)

selection_df = pd.read_csv(OUTPUTS_DIR / "family_model_selection.csv")
print("\nProduct Family Champion Model Selection:")
display(selection_df)

## 6. Continuous Demand Timeline & Prediction Intervals ($P10 / P50 / P90$)
Visualizing historical actual sales, reconstructed unconstrained demand, and the 19-month multi-quantile forecast with shaded uncertainty bands.

In [ ]:
from src.config import CLEANED_DATA_DIR, OUTPUTS_DIR
from src.dashboard.components import plot_continuous_timeline

# Load historical and forecast datasets
train_df = pd.read_parquet(CLEANED_DATA_DIR / "unconstrained_demand.parquet")
fcst_df = pd.read_parquet(OUTPUTS_DIR / "forecast_intervals.parquet")

# Aggregate monthly timelines
hist_monthly = train_df.groupby("Date").agg({
    "Units_Sold": "sum",
    "Units_Sold_Unconstrained": "sum"
}).reset_index()

fcst_monthly = fcst_df.groupby("Date").agg({
    "P10": "sum",
    "P50": "sum",
    "P90": "sum",
    "Units_Sold_Forecast": "sum"
}).reset_index()

# Render interactive Plotly timeline
fig_timeline = plot_continuous_timeline(
    hist_monthly,
    fcst_monthly,
    title="Celeris Automobili Continuous Demand Horizon (2019-2028) with P10/P90 Ribbon"
)
fig_timeline.show()

# Regional Forecast Breakdown
reg_df = fcst_df.groupby("Region")["Units_Sold_Forecast"].sum().reset_index()
fig_reg = px.pie(
    reg_df,
    names="Region",
    values="Units_Sold_Forecast",
    title="19-Month Global Forecast Distribution by Region",
    color_discrete_sequence=px.colors.sequential.Teal,
    template="plotly_dark",
    hole=0.4
)
fig_reg.update_layout(paper_bgcolor="#1E293B")
fig_reg.show()

## 7. Official Competition Submission Verification (58,219 Rows)
Comprehensive validation checks ensuring exact alignment with `student_data/test.csv`, non-negativity, zero nulls, and automated one-click file download in Google Colab.

In [ ]:
from src.config import RAW_TEST

sub_path = OUTPUTS_DIR / "submission.csv"
if not sub_path.exists():
    sub_path = Path("submission.csv")

sub_df = pd.read_csv(sub_path)
raw_test = pd.read_csv(RAW_TEST)

print("=" * 60)
print("  OFFICIAL SUBMISSION INTEGRITY CHECKLIST")
print("=" * 60)

# Check 1: Row count
assert len(sub_df) == 58219, f"Row count mismatch! Expected 58,219, got {len(sub_df)}"
print(f"✅ Row Count: {len(sub_df):,} matches test.csv exactly.")

# Check 2: Columns
expected_cols = ["Date", "SKU_ID", "Region", "Units_Sold_Forecast"]
assert list(sub_df.columns) == expected_cols, f"Column mismatch! Expected {expected_cols}, got {list(sub_df.columns)}"
print(f"✅ Columns: {list(sub_df.columns)} verified.")

# Check 3: Null count
null_cnt = sub_df.isnull().sum().sum()
assert null_cnt == 0, f"Found {null_cnt} null values!"
print(f"✅ Null Count: 0 missing or NaN values.")

# Check 4: Non-negativity
neg_cnt = (sub_df["Units_Sold_Forecast"] < 0).sum()
assert neg_cnt == 0, f"Found {neg_cnt} negative forecasts!"
print(f"✅ Validity: 100% non-negative integers.")

# Check 5: Alignment with test.csv
alignment_diff = (sub_df["Date"] != raw_test["Date"]).sum() + (sub_df["SKU_ID"] != raw_test["SKU_ID"]).sum()
assert alignment_diff == 0, f"Key alignment mismatch with test.csv in {alignment_diff} rows!"
print(f"✅ Alignment: 100% key-aligned with student_data/test.csv.")

print(f"\n📊 Total 19-Month Forecast Demand: {sub_df['Units_Sold_Forecast'].sum():,} units")
print(f"   Monthly Average: {sub_df.groupby('Date')['Units_Sold_Forecast'].sum().mean():,.0f} units/month")

display(sub_df.head(10))

# Colab One-Click Download
if IN_COLAB:
    from google.colab import files
    print("\n⬇️ Triggering one-click download of submission.csv to your local machine...")
    files.download(str(sub_path))

## 8. Launch Interactive Streamlit Control Tower in Google Colab
Launch the full 12-view executive S&OP Control Tower Dashboard (`app.py`) directly from Google Colab using `localtunnel`.

In [ ]:
# Launch Streamlit Control Tower in background and expose via localtunnel
if IN_COLAB:
    import urllib.request
    
    # 1. Start Streamlit in background
    !nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
    
    # 2. Get public IP address for localtunnel tunnel password
    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    except Exception:
        ip = "Check colab IP"
    print("=" * 70)
    print("🚀 STREAMLIT CONTROL TOWER LAUNCHED!")
    print(f"🔑 Your Tunnel Password (Enter this at the tunnel URL): {ip}")
    print("=" * 70)
    print("Click the link generated below to open the interactive dashboard:")
    
    # 3. Expose port 8501 via localtunnel with automatic yes to prompt
    !npx -y localtunnel --port 8501
else:
    print("💻 To run the dashboard locally, execute in your terminal:")
    print("   streamlit run app.py")